# Employee Churn — End-to-End Walkthrough

This notebook runs the full modeling workflow on a **reproducible synthetic dataset**, so it executes anywhere with no access to real HR records. It covers:

1. Synthetic data generation
2. Structured + text feature engineering
3. A multi-model zoo with cross-validation
4. Probability calibration
5. Group fairness diagnostics
6. SHAP explainability

In [ ]:
# Make the package importable when running from a fresh checkout
# (no installation required).
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'employee_churn').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from employee_churn.data import make_synthetic_employee_data

df = make_synthetic_employee_data(n=800, seed=42)
print(df.shape)
print('churn rate:', round(df['churned'].mean(), 3))
df.head()

## 1. Feature engineering

We combine career-progression, tenure-band, promotion-velocity, peer-relative compensation, sentiment, emotion, and text-shape features.

In [ ]:
from employee_churn.features.engineer_structured import (
    add_career_progression_features,
    add_tenure_bands,
    add_promotion_velocity,
    add_compensation_features,
    add_team_metrics,
)
from employee_churn.features.engineer_text import add_text_statistics
from employee_churn.nlp.sentiment import add_sentiment_scores
from employee_churn.nlp.emotion import add_emotion_features

feat = add_career_progression_features(df, 'hire_date', 'last_promotion_date')
feat = add_tenure_bands(feat)
feat = add_promotion_velocity(feat, 'num_promotions')
feat = add_compensation_features(feat, 'monthly_salary', 'department')
feat = add_team_metrics(feat, 'team_id')
feat = add_sentiment_scores(feat, 'feedback')
feat = add_emotion_features(feat, 'feedback')
feat = add_text_statistics(feat, 'feedback')
feat.filter(regex='sentiment|emotion_|text_|salary_|promotions_per').head()

## 2. Build the modeling matrix

Keep numeric/boolean columns, drop identifiers and raw dates, and one-hot encode the remaining low-cardinality categoricals.

In [ ]:
sensitive = feat['gender'].copy()
target = feat['churned'].copy()

drop_cols = [
    'employee_id', 'churned', 'feedback', 'gender', 'department',
    'hire_date', 'last_promotion_date', 'team_id', 'tenure_band',
    'emotion_dominant',
]
X = feat.drop(columns=drop_cols).select_dtypes(include=['number', 'bool'])
y = target
print(X.shape)
X.head()

## 3. Model zoo + cross-validation

Compare every classifier with stratified 5-fold cross-validation.

In [ ]:
from employee_churn.models.train import build_model_zoo, cross_validate_models

zoo = build_model_zoo(random_state=0)
cv_results = cross_validate_models(zoo, X, y, cv=5)
pd.DataFrame(cv_results).T.sort_values('roc_auc_mean', ascending=False)

## 4. Hyperparameter tuning + calibration

Tune the strongest tree model, then check whether calibration improves the reliability of its probabilities.

In [ ]:
from sklearn.model_selection import train_test_split
from employee_churn.models.train import tune_hyperparameters
from employee_churn.models.calibrate import calibration_improvement

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)
best, params, score = tune_hyperparameters(
    zoo['random_forest'], X_tr, y_tr, model_name='random_forest', n_iter=5
)
print('best params:', params, 'cv roc_auc:', round(score, 3))

calib = calibration_improvement(best, X_tr, y_tr, X_te, y_te)
print('ECE before:', round(calib['baseline']['expected_calibration_error'], 4))
print('ECE after :', round(calib['calibrated']['expected_calibration_error'], 4))
print('improved  :', calib['improved'])

## 5. Fairness diagnostics

Check selection-rate parity across the (synthetic) gender attribute using the four-fifths rule.

In [ ]:
from employee_churn.models.fairness import (
    group_fairness_report,
    fairness_summary,
)

preds = best.fit(X_tr, y_tr).predict(X_te)
report = group_fairness_report(y_te.values, preds, sensitive.loc[y_te.index].values)
display(report)
fairness_summary(report)

## 6. Explainability

SHAP attributions for a sample of employees.

In [ ]:
from employee_churn.models.explain import explain_with_shap

sample = X_te.head(50)
shap_df = explain_with_shap(best, sample)
shap_df.abs().mean().sort_values(ascending=False).head(10)